<a href="https://colab.research.google.com/github/liyhzzzzz-lang/lilili/blob/main/PSICHIC3.10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# $\text{PSICHIC}_{\mathbb{V1.0}} \text{Virtual Screening Platform on Google Colab}$

| [Preprint](https://doi.org/10.1101/2023.09.17.558145) | [Github Repo](https://github.com/huankoh/PSICHIC) | [Run in Colab](https://colab.research.google.com/github/huankoh/PSICHIC/blob/main/PSICHIC.ipynb) (click `Runtime` → `Run all (Ctrl+F9)` |

## PSICHIC Features

- **Quick Screening**: Up to 100K compounds in an hour.
- **Deep Analysis**: Uncover molecular insights with PSICHIC's unique algorithms.
- **More Tools**: Pharmacophore and targeted mutagenesis analysis.

## Input Needed

- **Only Sequence Data**: Protein Sequence + Ligand SMILES pairs.

## Why PSICHIC?

- **Fast**: Save time on initial screenings.
- **Smart**: Make data-driven decisions.
- **Efficient**: No special hardware needed.

Start exploring. Your next discovery is just clicks away!

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%capture
# @title Setting up PSICHIC Online Platform (~ 7 min)
import os, time

# Uninstall existing packages to prevent conflicts
# Using -y to automatically confirm uninstallation, and redirecting output to /dev/null
print("Uninstalling existing PyTorch and PyTorch Geometric components...")
!pip uninstall -y torch torchvision torchaudio > /dev/null 2>&1
!pip uninstall -y torch-geometric pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv > /dev/null 2>&1
print("Uninstallation complete.")

# 1) Install PyTorch with specific version and CUDA compatibility
print("Installing torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 for cu121...")
!pip install torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 --index-url https://download.pytorch.org/whl/cu121 > /dev/null 2>&1
print("PyTorch installation complete.")

# 2) Install PyTorch Geometric and its dependencies
# The correct find-links URL for PyG depends on the PyTorch and CUDA versions.
# Since we explicitly installed torch 2.3.0 with cu121, we can use that.
TORCH_VERSION_STR = "2.3.0"
CUDA_VERSION_STR = "cu121"
print(f"Installing PyTorch Geometric and dependencies for torch-{TORCH_VERSION_STR}+{CUDA_VERSION_STR}...")
!pip install torch_geometric > /dev/null 2>&1
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-{TORCH_VERSION_STR}+{CUDA_VERSION_STR}.html > /dev/null 2>&1
print("PyTorch Geometric installation complete.")


# 3) Core dependencies (always upgrade to latest compatible)
print("Installing core dependencies...")
deps = [
    "rdkit", "biopython", "biopandas", "py3Dmol", "timeout_decorator",
    "cairosvg", "umap-learn", "plotly", "mplcursors",
    "lifelines", "reprint",
]
os.system("pip install -U " + " ".join(deps) + " > /dev/null 2>&1")
print("Core dependencies installation complete.")


# 4) Clone PSICHIC (latest from GitHub main)
print("Cloning PSICHIC repository...")
os.system("git clone https://github.com/huankoh/PSICHIC.git > /dev/null 2>&1")
print("PSICHIC repository cloned.")

version = "1"
model_name = "esmfold_v0.model" if version == "0" else "esmfold.model"
import os, time
if not os.path.isfile(model_name):
  # download esmfold params
  print("ESMFold model not found, downloading...")
  os.system("apt-get install aria2 -qq > /dev/null 2>&1")
  os.system(f"aria2c -q -x 16 https://colabfold.steineggerlab.workers.dev/esm/{model_name} & ")

  if not os.path.isfile("finished_install"):
    # install libs
    print("Installing additional ESMFold related libs...")
    os.system("pip install -q omegaconf pytorch_lightning biopython ml_collections einops py3Dmol modelcif > /dev/null 2>&1")
    os.system("pip install -q git+https://github.com/NVIDIA/dllogger.git > /dev/null 2>&1")

    print("Installing openfold...")
    # install openfold
    os.system(f"pip install -q git+https://github.com/sokrypton/openfold.git > /dev/null 2>&1")

    print("Installing esmfold...")
    # install esmfold
    os.system(f"pip install -q git+https://github.com/sokrypton/esm.git > /dev/null 2>&1")
    os.system("touch finished_install")

  # wait for Params to finish downloading...
  while not os.path.isfile(model_name):
    time.sleep(5)
  if os.path.isfile(f"{model_name}.aria2"):
    print("Downloading params...")
  while os.path.isfile(f"{model_name}.aria2"):
    time.sleep(5)
  print("ESMFold model download complete.")

import pandas as pd
from IPython.display import SVG
from rdkit import Chem
import numpy as np

%cd PSICHIC
from rdkit import RDLogger

# Disable all RDKit logs (errors, warnings, info)
RDLogger.DisableLog('rdApp.*')
import warnings

warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning,
    module="rdkit"
)

In [31]:
#@title 1. Importing functions
import matplotlib.pyplot as plt
from matplotlib.colors import rgb2hex
from ipywidgets import Button, Layout, jslink, IntText, IntSlider
import io
import IPython
from ipywidgets import GridspecLayout
from ipywidgets import Output

from matplotlib.colors import LinearSegmentedColormap
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

import Bio.PDB as PDB

import py3Dmol


from rdkit.Chem.Draw import SimilarityMaps
from rdkit.Chem import Draw
import pickle

from string import ascii_uppercase, ascii_lowercase
import hashlib, re, os
import numpy as np
import torch
from jax.tree_util import tree_map
import matplotlib.pyplot as plt
from scipy.special import softmax
import gc
import warnings
from rdkit.Chem import AllChem

from rdkit.Chem import rdMolAlign
import timeout_decorator
@timeout_decorator.timeout(600)  # Set the timeout to 10 minutes (600 seconds)


def read_molecule(molecule_file, sanitize=False, calc_charges=False, remove_hs=True):
    if molecule_file.endswith('.mol2'):
        mol = Chem.MolFromMol2File(molecule_file, sanitize=False, removeHs=False)
    elif molecule_file.endswith('.sdf'):
        supplier = Chem.SDMolSupplier(molecule_file, sanitize=False, removeHs=False)
        mol = supplier[0]
    elif molecule_file.endswith('.pdbqt'):
        with open(molecule_file) as file:
            pdbqt_data = file.readlines()
        pdb_block = ''
        for line in pdbqt_data:
            pdb_block += '{}\n'.format(line[:66])
        mol = Chem.MolFromPDBBlock(pdb_block, sanitize=False, removeHs=False)
    elif molecule_file.endswith('.pdb'):
        mol = Chem.MolFromPDBFile(molecule_file, sanitize=False, removeHs=False)
    else:
        raise ValueError('Expect the format of the molecule_file to be '
                         'one of .mol2, .sdf, .pdbqt and .pdb, got {}'.format(molecule_file))

    try:
        if sanitize or calc_charges:
            Chem.SanitizeMol(mol)

        if calc_charges:
            # Compute Gasteiger charges on the molecule.
            try:
                AllChem.ComputeGasteigerCharges(mol)
            except:
                warnings.warn('Unable to compute charges for the molecule.')

        if remove_hs:
            mol = Chem.RemoveHs(mol, sanitize=sanitize)

    except Exception as e:
        # print(e)
        return None

    return mol


def read_mol(mol_path, remove_hs=False):
    lig = None
    if mol_path.lower().endswith('.sdf'):
        lig = read_molecule(mol_path, remove_hs=remove_hs, sanitize=True)
    elif mol_path.lower().endswith('.mol2'):
        lig = read_molecule(mol_path, remove_hs=remove_hs, sanitize=True)
    elif mol_path.lower().endswith('.pdb'):
        # could set sanitize=False if peptides cause issues
        lig = read_molecule(mol_path, remove_hs=remove_hs, sanitize=True)

    if lig is None:
        raise Exception(f"Fail to read ligand path: {mol_path}")

    return lig




show_sidechains=False

mapper = plt.get_cmap('Reds', )
color_map = {}


one_to_three = {"A" : "ALA",
              "C" : "CYS",
              "D" : "ASP",
              "E" : "GLU",
              "F" : "PHE",
              "G" : "GLY",
              "H" : "HIS",
              "I" : "ILE",
              "K" : "LYS",
              "L" : "LEU",
              "M" : "MET",
              "N" : "ASN",
              "P" : "PRO",
              "Q" : "GLN",
              "R" : "ARG",
              "S" : "SER",
              "T" : "THR",
              "V" : "VAL",
              "W" : "TRP",
              "Y" : "TYR",
              "B" : "ASX",
              "Z" : "GLX",
              "X" : "UNK",
              "*" : " * "}

three_to_one = {}
for _key, _value in one_to_three.items():
    three_to_one[_value] = _key
three_to_one["SEC"] = "C"
three_to_one["MSE"] = "M"
def overwrite_b_factors(structure,
                        custom_bfactors: np.ndarray, res_symbols: list,
                        save_path=None) -> str:

    curr_resid = ('', '', '')
    idx = -1
    a_list = []
    for i, chain in enumerate(structure):
        for res_idx, residue in enumerate(chain):
            if residue.get_resname() == 'HOH':
                for atom in residue:
                    atom.bfactor = 0

            c_alpha, n, c = None, None, None
            for atom in residue:
                if atom.name == 'CA':
                    c_alpha = list(atom.get_vector())
                if atom.name == 'N':
                    n = list(atom.get_vector())
                if atom.name == 'C':
                    c = list(atom.get_vector())
            if c_alpha != None and n != None and c != None:
                idx += 1

                for atom in residue:
                    ##### ensure visualization SEQ follows pre-processed SEQ
                    try:
                        pdb_res_symbol = three_to_one[residue.get_resname()]
                    except Exception as e:
                        pdb_res_symbol = 'X'
                        # print("encountered unknown AA: ", residue.get_resname(), ' in the complex. Replacing it with a dash X.')

                    if res_symbols[idx] != pdb_res_symbol:
                        raise Exception('Result sequence symbol {} does not match with PDB sequence symbol {}'.format(res_symbols[idx],pdb_res_symbol))

                    atom.bfactor = custom_bfactors[idx]
            else:
                for atom in residue:
                    atom.bfactor = 0
    if save_path is None:
        path_out = 'temporary.txt'
    else:
        path_out = save_path
    new_pdb = io.StringIO()
    pdb_io = PDB.PDBIO()
    pdb_io.set_structure(structure)
    pdb_io.save(path_out)

    with open(path_out,'r') as f:
        out = f.read()

    return out

for v in range(100+1):
    rgba = mapper(v/100)
    color_map[v/100] = rgb2hex(rgba)

def DrawDocking(protein,ligand=None,
                show_protein_surface=True, show_protein_mainchain=False,show_protein_sidechain=False):
    viewer = py3Dmol.view(width=800,height=800)
    viewer.addModelsAsFrames(protein)

    if show_protein_sidechain:
        BB = ['C','O','N']
        viewer.addStyle({'and':[{'resn':["GLY","PRO"],'invert':True},{'atom':BB,'invert':True}]},
                    {'stick':{'colorscheme': {'prop': 'b', 'map': color_map},'radius':0.3}})
        viewer.addStyle({'and':[{'resn':"GLY"},{'atom':'CA'}]},
                    {'sphere':{'colorscheme': {'prop': 'b', 'map': color_map},'radius':0.3}})
        viewer.addStyle({'and':[{'resn':"PRO"},{'atom':['C','O'],'invert':True}]},
                    {'stick':{'colorscheme': {'prop': 'b', 'map': color_map},'radius':0.3}})

    if show_protein_mainchain:
        BB = ['C','O','N','CA']
        viewer.addModelsAsFrames
        viewer.addStyle({'atom':BB},{'stick':{'colorscheme': {'prop': 'b', 'map': color_map},'radius':0.3}})

    if show_protein_surface:
        BB = ['C','O','N','CA']
        prot = {'resn': ["DMS", "UNL", "SO4", "LIG", "HOH", "Cl"], 'invert': 1}  #define prot as all except list
        style = {'cartoon': {'colorscheme': {'prop': 'b', 'map': color_map}}}
        viewer.addSurface(py3Dmol.VDW,{'opacity':0.8, 'colorscheme': {'prop': 'b', 'map': color_map}}, prot)

    if ligand is not None:
        # viewer.addModel(ligand)
        # MyLig = {'resn':'UNL'}
        # viewer.setStyle(MyLig,{'stick':{'colorscheme' : 'greenCarbon'}})
        # viewer.zoomTo(MyLig)

        # explicitly say this is PDB
        viewer.addModel(ligand, 'pdb')
        # style the last-added model (model: -1 means "last model")
        viewer.setStyle({'model': -1}, {'stick': {'colorscheme': 'greenCarbon'}})
        viewer.zoomTo({'model': -1})
    else:
        viewer.zoomTo()
    return viewer

def alphafold_plot(to_visualize_pdb):
    view = py3Dmol.view(width=800, height=600)
    view.addModelsAsFrames(to_visualize_pdb)
    style = {'cartoon': {'colorscheme': {'prop': 'b', 'map': color_map}}}
    if show_sidechains:
        style['stick'] = {}
    view.setStyle({'model': -1}, style)
    view.zoomTo()
    return view

def generate_ESM_structure(jobname, sequence):
    def get_hash(x): return hashlib.sha1(x.encode()).hexdigest()
    alphabet_list = list(ascii_uppercase+ascii_lowercase)

    jobname = re.sub(r'\W+', '', jobname)[:50]

    sequence = re.sub("[^A-Z:]", "", sequence.replace("/",":").upper())
    sequence = re.sub(":+",":",sequence)
    sequence = re.sub("^[:]+","",sequence)
    sequence = re.sub("[:]+$","",sequence)
    copies = 1
    sequence = ":".join([sequence] * copies)
    num_recycles = 3
    chain_linker = 25

    ID = jobname+"_"+get_hash(sequence)[:5]
    seqs = sequence.split(":")
    lengths = [len(s) for s in seqs]
    length = sum(lengths)
    print("length",length)

    u_seqs = list(set(seqs))
    if len(seqs) == 1: mode = "mono"
    elif len(u_seqs) == 1: mode = "homo"
    else: mode = "hetero"

    if "model" not in dir() or model_name != model_name_:
        if "model" in dir():
            # delete old model from memory
            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        model = torch.load(model_name)
        model.eval().cuda().requires_grad_(False)
        model_name_ = model_name

    # optimized for Tesla T4
    if length > 700:
        model.set_chunk_size(64)
    else:
        model.set_chunk_size(128)

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    output = model.infer(sequence,
                        num_recycles=num_recycles,
                        chain_linker="X"*chain_linker,
                        residue_index_offset=512)

    pdb_str = model.output_to_pdb(output)[0]

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return pdb_str


def plot_visual(protein_csv, ligand_pickle,  protein_path=None, ligand_path=None, save_id='',
                residue_color_type='Original', show_protein_surface=True,
                show_protein_mainchain=False,show_protein_sidechain=False):
    protein_df = pd.read_csv(protein_csv)
    if residue_color_type == 'Original':
        residue_scores = protein_df['PSICHIC_Residue_Score'].round(2).clip(0.01,0.99).tolist()
    else:
        residue_scores = protein_df['PSICHIC_Residue_Percentile'].round(2).clip(0.01,0.99).tolist()

    if protein_path:
        parser = PDB.PDBParser(QUIET=True)
        structure = parser.get_structure('', protein_path)[0]
        protein_pdb_block = overwrite_b_factors(structure, residue_scores, protein_df['Residue_Type'].tolist())
    else:
        print('No PDB detected - generating in silico structure from ESM')
        pdbstr = generate_ESM_structure(save_id + '_ESM.pdb', ''.join(protein_df['Residue_Type'].tolist()))
        torch.cuda.empty_cache()

        parser = PDB.PDBParser(QUIET=True)
        handle = io.StringIO(pdbstr)
        structure = parser.get_structure('', handle)[0]
        protein_pdb_block = overwrite_b_factors(structure, residue_scores, protein_df['Residue_Type'].tolist())


    ligand_pdb_block = None
    if ligand_path:

        # use your RDKit path (sdf / mol2 etc.)
        ligand = read_mol(ligand_path)
        if ligand is None:
            raise ValueError(f"Failed to read ligand from {ligand_path}")
        ligand_pdb_block = Chem.MolToPDBBlock(ligand)


    view = DrawDocking(protein_pdb_block, ligand_pdb_block,
                       show_protein_surface=show_protein_surface,
                        show_protein_mainchain=show_protein_mainchain,
                        show_protein_sidechain=show_protein_sidechain)
    grid = GridspecLayout(1,2, layout=Layout(width='80%', height='80%'))
    out = Output()
    with out:
        view.show()
    grid[0, 0] = out

    ## atom scores
    with open(ligand_pickle,'rb') as f:
        mol = pickle.load(f)

    ligand_contribs = []
    for idx, atom in enumerate(mol.GetAtoms()):
        psichic_atom_score = atom.GetProp("PSICHIC_Atom_Score")
        atom.SetProp('score',psichic_atom_score)
        ligand_contribs.append(float(psichic_atom_score.replace('[','').replace(']','')))


    d1 = Draw.MolDraw2DSVG(500, 500)
    def make_darker(color, scale):
        r, g, b, a = color
        return (r * scale, g * scale, b * scale, a)

    start_color = mcolors.hex2color('#40d0e0')
    middle_color = mcolors.hex2color('white')
    end_color = mcolors.hex2color('#40d0e0')
    cmap = mcolors.LinearSegmentedColormap.from_list(
        'custom_cmap', [start_color, middle_color, end_color])
    darker_first_color = make_darker(cmap(0), 0.8)
    darker_last_color = make_darker(cmap(1), 0.8)

    colorMap = LinearSegmentedColormap.from_list('Wistia', [darker_first_color, (1.0, 1.0, 1.0), darker_last_color], N=255)
    SimilarityMaps.GetSimilarityMapFromWeights(mol, ligand_contribs, draw2d=d1, contourLines=10, colorMap=colorMap)
    d1.FinishDrawing()
    svg1 = SVG(d1.GetDrawingText())

    out = Output()
    with out:
        IPython.display.display(svg1)
    grid[0,1] = out
    IPython.display.display(grid)

    return {'protein_pdb_block':protein_pdb_block, 'protein_path':protein_path}, {'ligand_pdb_block':ligand_pdb_block,'ligand2d':mol, 'atom_scores':ligand_contribs, 'ligand_path':ligand_path}


# PSICHIC Sequence-based Library Screening

- User-Friendly: Just a CSV file with ``Protein`` and ``Ligand`` columns needed. It is desriable to also include an ``ID`` column to refer each unique pair during your analysis at later stage. If ``ID`` is missing, we will include an ID based on the row number (PAIR_1, ..., PAIR_N).

Your CSV should look something like this:

| ID | Protein | Ligand |
|:----------:|:----------:|:----------:|
| MIPS1| ATCGATCG....  | C1CCCCC1  |
| MIPS2| GCTAGCTA....  | O=C(C)Oc1ccccc1C(=O)O |
|...|...| ...|
|MIPS100K|TACGTACG | CCO |

- Fast & Accurate: Get reliable results quickly.
- Streamlined Workflow: All-in-one solution for protein-ligand interaction
analysis.

In [9]:
#@title 1. Upload your screening csv file here ⬇️

#@markdown Running this code block, will prompt you to upload a csv file for virtual screening. If you just want to try PSICHIC, we have a file for PSICHIC demo, just click 'Cancel upload'.

#@markdown Specify the name of the result folder below (job_id):
jobname = "PSICHIC_RESULT" #@param {type:"string"}

import io
import pandas as pd
from google.colab import files


try:
    uploaded = files.upload()
    filename=[key for key in uploaded.keys()][0]
    df = pd.read_csv(io.BytesIO(uploaded[filename]))
except:
    print('Fail to read any uploaded files. Using our demo file from PDB v2020 test data instead..')
    filename = 'dataset/pdb2020/test.csv'

Saving psichic_input.csv to psichic_input (1).csv


In [10]:
#@title 2. Run screening using PSICHIC
#@markdown By default, we will use the PSICHIC model trained on PDBBind v2020. This is the model that is most comprehensively evaluated (including its interpretability). For application purposes, you should choose the PSICHIC-MultiTask model where we pre-trained it on Large-scale Interaction Database.
PSICHIC_parameters = "Pre-trained on Large-scale Interaction Database" #@param ["Trained on PDBBind v2020", "Pre-trained on Large-scale Interaction Database"]
#@markdown By default, we will also save the interpretation from PSICHIC. If you only want the prediction output without further analysis, you can turn this off. However, in this case, you will not be able to use the rest of the notebook for analysis.
Save_PSICHIC_Interpretation = True #@param {type:"boolean"}
#@markdown By default, we use batch size of 16. In most cases, this is a good default - You may need to decrease it if the protein sequence length is very long.
batch_size = 16 #@param {type:"integer"}
if PSICHIC_parameters == 'Trained on PDBBind v2020':
    trained_model_path = 'trained_weights/PDBv2020_PSICHIC'
elif PSICHIC_parameters == 'Pre-trained on Large-scale Interaction Database':
    trained_model_path = 'trained_weights/multitask_PSICHIC'

screenfile = filename
result_path = jobname
#@markdown Three substep you will observe is (1) Initialising Protein, (2) Initialising Ligand and (3) Start Screening.

import json
import pandas as pd
import torch
import numpy as np
import os
import random
# Utils
from utils.utils import DataLoader, virtual_screening
from utils.dataset import *  # data
from utils.trainer import Trainer
from utils.metrics import *
# Preprocessing
from utils import protein_init, ligand_init
# Model
from models.net import net
# Config
from tqdm import tqdm

device = 'cuda:0'
with open(os.path.join(trained_model_path,'config.json'),'r') as f:
    config = json.load(f)

print("Screening the csv file: {}".format(screenfile))
# device
device = torch.device(device)


if not os.path.exists(result_path):
    os.makedirs(result_path)

degree_dict = torch.load(os.path.join(trained_model_path,'degree.pt'))
param_dict = os.path.join(trained_model_path,'model.pt')
mol_deg, prot_deg = degree_dict['ligand_deg'],degree_dict['protein_deg']

model = net(mol_deg, prot_deg,
            # MOLECULE
            mol_in_channels=config['params']['mol_in_channels'],  prot_in_channels=config['params']['prot_in_channels'],
            prot_evo_channels=config['params']['prot_evo_channels'],
            hidden_channels=config['params']['hidden_channels'], pre_layers=config['params']['pre_layers'],
            post_layers=config['params']['post_layers'],aggregators=config['params']['aggregators'],
            scalers=config['params']['scalers'],total_layer=config['params']['total_layer'],
            K = config['params']['K'],heads=config['params']['heads'],
            dropout=config['params']['dropout'],
            dropout_attn_score=config['params']['dropout_attn_score'],
            # output
            regression_head=config['tasks']['regression_task'],
            classification_head=config['tasks']['classification_task'] ,
            multiclassification_head=config['tasks']['mclassification_task'],
            device=device).to(device)
model.reset_parameters()
model.load_state_dict(torch.load(param_dict,map_location=device))


screen_df = pd.read_csv(os.path.join(screenfile))
protein_seqs = screen_df['Protein'].unique().tolist()
print('Initialising protein sequence to Protein Graph')
protein_dict = protein_init(protein_seqs)
ligand_smiles = screen_df['Ligand'].unique().tolist()
print('Initialising ligand SMILES to Ligand Graph')
ligand_dict = ligand_init(ligand_smiles)
torch.cuda.empty_cache()
## drop any invalid smiles
screen_df = screen_df[screen_df['Ligand'].isin(list(ligand_dict.keys()))].reset_index(drop=True)
screen_dataset = ProteinMoleculeDataset(screen_df, ligand_dict, protein_dict, device=device)
screen_loader = DataLoader(screen_dataset, batch_size=batch_size, shuffle=False,
                            follow_batch=['mol_x', 'clique_x', 'prot_node_aa'])

print("Screening starts now!")
screen_df = virtual_screening(screen_df, model, screen_loader,
                 result_path=os.path.join(result_path, "interpretation_result"), save_interpret=Save_PSICHIC_Interpretation,
                 ligand_dict=ligand_dict, device=device)

screen_df.to_csv(os.path.join(result_path,'screening.csv'),index=False)
print('Screening completed and saved to {}'.format(result_path))

Screening the csv file: psichic_input (1).csv
Initialising protein sequence to Protein Graph


  0%|          | 0/5 [00:00<?, ?it/s]

Initialising ligand SMILES to Ligand Graph


  0%|          | 0/19 [00:00<?, ?it/s]

Screening starts now!


  0%|          | 0/6 [00:00<?, ?it/s]

Screening completed and saved to PSICHIC_RESULT


In [11]:
#@title 3. OPTIONAL - Download Screening Results
#@markdown If you are having issues downloading the result archive, try disabling your adblocker and run this cell again. If that fails click on the little folder icon to the left, navigate to file: `jobname.result.zip`, right-click and select \"Download\" (see [screenshot](https://pbs.twimg.com/media/E6wRW2lWUAEOuoe?format=jpg&name=small)).

#@title Do you agree? { run: "auto" }
download_result = True #@param {type:"boolean"}

if download_result:
    os.system(f"zip -r {jobname}.result.zip {jobname}" )
    files.download(f"{jobname}.result.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# PSICHIC Interactive Platform for Drug Discovery

This interactive UMAP scatter plot visualizes compounds based on their binding affinity and the probability of being an agonist, antagonist, or non-binder. This is a powerful tool for drug discovery, enabling you to quickly identify potential drug candidates and understand the chemical space.


#### Visual Encoding:

- **Size and Color**: The size of each point corresponds to its binding affinity. Larger points (redder color) indicate higher binding affinity.
- **Shape**: Points take different shapes based on whether they are most likely to be agonists, antagonists, or non-binders.

#### How to Explore:

1. **Hover Over Points**: Move your cursor over individual points to reveal their unique IDs, binding affinity, and agonist/antagonist/non-binder probability.
2. **Zoom and Pan**: Use your mouse wheel to zoom in and out. Click and drag to pan across different regions of the plot.
3. **Identify Clusters**: Look for areas where points are densely packed together. These could represent compounds with similar mechanisms of action or structural features.
4. **Note Outliers**: Pay attention to points that are far away from any cluster. These could be novel compounds with unique properties worth investigating.

#### Let your curiosity guide you in uncovering the next blockbuster drug!
- **High-Affinity Clusters**: Are there clusters of compounds with high binding affinity? These could be strong candidates for drug development.
- **Agonist/Antagonist Patterns**: Do you notice any trends related to the probability of being an agonist or antagonist? This could help in targeting specific pathways.
- **Anomalies and Novel Compounds**: Are there any points far away from any clusters? These could be novel scaffolds or unique mechanisms of action that are worth exploring.


In [4]:
#@title 📂 导入 PSICHIC 结果数据 (注：数据更新后需重新初始化交互平台)

import os
import zipfile
from google.colab import files

print("👇 请在下方上传您的结果压缩包 (例如 PSICHIC_RESULT.zip):")
# 唤起本地文件上传窗口
uploaded = files.upload()

if uploaded:
    # 获取上传的压缩包文件名
    zip_filename = list(uploaded.keys())[0]
    print(f"\n成功上传: {zip_filename}")

    # 修改：设定解压路径为 /content/PSICHIC
    extract_path = '/content/PSICHIC'

    # 确保目标文件夹存在，避免因目录不存在报错
    os.makedirs(extract_path, exist_ok=True)

    print(f"正在解压文件至 {extract_path}，请稍候...")
    # 执行解压操作
    with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

    print("✅ 解压完成！")

    # 修改：验证解压后的新路径是否正确包含核心结果文件
    expected_csv_path = '/content/PSICHIC/PSICHIC_RESULT/screening.csv'

    if os.path.exists(expected_csv_path):
        print(f"🎉 路径就绪：已检测到核心结果文件 {expected_csv_path}")
        print("👉 现在您可以直接向下滚动，运行笔记本自带的「Interactive Platform」可视化代码块了。")
    else:
        print(f"⚠️ 解压成功，但在预期路径 {expected_csv_path} 未找到 screening.csv。")
        print("如果您的压缩包内多嵌套了一层文件夹，请手动在左侧文件栏中核对并移动文件。")
else:
    print("未上传任何文件，操作取消。")

👇 请在下方上传您的结果压缩包 (例如 PSICHIC_RESULT.zip):


Saving PSICHIC_RESULT.result (8).zip to PSICHIC_RESULT.result (8).zip

成功上传: PSICHIC_RESULT.result (8).zip
正在解压文件至 /content/PSICHIC，请稍候...
✅ 解压完成！
🎉 路径就绪：已检测到核心结果文件 /content/PSICHIC/PSICHIC_RESULT/screening.csv
👉 现在您可以直接向下滚动，运行笔记本自带的「Interactive Platform」可视化代码块了。


In [5]:
#@title 1. Data Setup for Interactive Platform
from tqdm.notebook import tqdm
import umap
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt

# Fix: Define result_path if it's not already defined
# This assumes jobname has been set in a previous cell, which is 'PSICHIC_RESULT' based on the notebook state.
if 'result_path' not in locals():
    result_path = 'PSICHIC_RESULT'

df = pd.read_csv(os.path.join(result_path,'screening.csv'))

fps = []
for idx, row in tqdm(df.iterrows(), total=df.shape[0]):
    fp = np.load(os.path.join(result_path,'interpretation_result',row['ID'],'fingerprint.npy'))
    fps.append(fp)

fps = np.stack(fps)
reducer = umap.UMAP()
embedding = reducer.fit_transform(fps)
df[['x','y']] = embedding


  0%|          | 0/95 [00:00<?, ?it/s]

In [6]:
#@title 2. Interactive Platform for Drug Discovery

import plotly.graph_objects as go

#@markdown Scaling factor will adjust the size of each dot. Adjust the figure width and height for optimal visual.
scaling_factor = 1.5 #@param {type:"number"}
figure_width = 1500 #@param {type:"integer"}
figure_height = 1000 #@param {type:"integer"}

df[['x','y']] = embedding
df = df.rename(columns={'predicted_agonist':'Agonist Probability',
                'predicted_antagonist':'Antagonist Probability',
                'predicted_nonbinder':'Non-binder Probability',
                'predicted_binding_affinity':'Binding Affinity'})

hover_data_list = ['ID','Binding Affinity']

def exponential_scale(value, base=2):
    return 4 + (base ** (value - 4))
def functional_effect_shape(cls):
    if cls == 'Agonist':
        return 'triangle-up'
    elif cls == 'Antagonist':
        return 'triangle-down'
    else:
        return 'circle'

df['Binding Affinity Size'] = exponential_scale(df['Binding Affinity'].clip(4,10)) * scaling_factor
if 'Agonist Probability' in df.columns and 'Antagonist Probability' in df.columns and 'Non-binder Probability' in df.columns:
    # Determine the shape based on the highest probability category
    df['Functional Effect Class'] = df[['Agonist Probability', 'Antagonist Probability', 'Non-binder Probability']].idxmax(axis=1)
    df['Functional Effect Class'] = df['Functional Effect Class'].apply(lambda x: x.replace(' Probability',''))
    df['Shape'] = df['Functional Effect Class'].apply(functional_effect_shape)
    hover_data_list += ['Agonist Probability', 'Antagonist Probability', 'Non-binder Probability']


hover_string = "ID: %{customdata[0]}<br>Binding Affinity: %{customdata[1]:.4f}"
if 'Shape' in df.columns:
    hover_string += "<br>Agonist Probability: %{customdata[2]:.4f}<br>Antagonist Probability: %{customdata[3]:.4f}<br>Non-binder Probability: %{customdata[4]:.4f}"

# Customize color bar
max_value = df['Binding Affinity Size'].max()
min_value = df['Binding Affinity Size'].min()
ori_max_value = df['Binding Affinity'].max()
ori_min_value = df['Binding Affinity'].min()


fig = go.Figure()

if 'Functional Effect Class' in df.columns:
    # Loop through each unique class to create a separate trace
    for cls in df['Functional Effect Class'].unique():
        subset_df = df[df['Functional Effect Class'] == cls]
        fig.add_trace(go.Scatter(
            x=subset_df['x'],
            y=subset_df['y'],
            mode='markers',
            marker=dict(
                size=subset_df['Binding Affinity Size'],
                color=subset_df['Binding Affinity Size'],
                colorscale=px.colors.sequential.Redor,
                colorbar=dict(
                    title="Binding Affinity Size",
                    tickvals=[min_value, max_value],
                    ticktext=[f"Affinity Min: {ori_min_value}", f"Affinity Max: {ori_max_value}"]
                ),
                symbol=subset_df['Functional Effect Class'].apply(lambda x: functional_effect_shape(x))
            ),
            name=cls,  # This name will appear in the legend
            customdata=subset_df[hover_data_list],
            hovertemplate=hover_string
        ))

    # Customize hover template
    fig.update_traces(
        hovertemplate=hover_string
    )

    fig.update_layout(
        legend=dict(
            x=0,  # Adjust as needed
            y=1.1,
            traceorder="normal",
            orientation="h",
            font=dict(
                size=15
            )

        ),
        width=figure_width,  # Width in pixels
        height=figure_height # Height in pixels
    )
else:
    fig.add_trace(go.Scatter(
            x=df['x'],
            y=df['y'],
            mode='markers',
            marker=dict(
                size=df['Binding Affinity Size'],
                color=df['Binding Affinity Size'],
                colorscale=px.colors.sequential.Redor,
                colorbar=dict(
                    title="Binding Affinity Size",
                    tickvals=[min_value, max_value],
                    ticktext=[f"Affinity Min: {ori_min_value}", f"Affinity Max: {ori_max_value}"]
                ),
            ),
            customdata=df[hover_data_list],
            name='',  # Set name to an empty string
            hovertemplate=hover_string
        ))

    # Customize hover template
    fig.update_traces(
        hovertemplate=hover_string
    )
fig.show()


# PSICHIC's Interpretation of Protein-Ligand Interaction

In [29]:
#@title 2. Visualisation of PSICHIC Importance Scores（带 TOP5 防重叠 + 增强版下载器）
import os
import pandas as pd
import py3Dmol
from IPython.display import display, HTML

pair_id = "LYH000088" #@param {type:"string"}
protein_path='' #@param {type:"string"}
ligand_path= '' #@param {type:"string"}

if not protein_path:
    protein_seq = df[df['ID'] == pair_id]['Protein'].item()
    print('protein sequence length:', len(protein_seq))
    if len(protein_seq) > 700:
        raise Exception('Protein sequence length is greater than 700, this will cause out-of-memory error. Please use separate tools to get in silico structures.')

residue_color = "Percentile Ranking" #@param ["Original", "Percentile Ranking"]
show_protein_mainchain = True #@param {type:"boolean"}
show_protein_surface = True #@param {type:"boolean"}
show_protein_sidechain = False #@param {type:"boolean"}

protein_interpret = os.path.join(result_path,'interpretation_result',pair_id,'protein.csv')
ligand_interpret = os.path.join(result_path,'interpretation_result',pair_id,'ligand.pkl')

# =================================================================
# 核心黑科技：底层渲染引擎拦截器 (注入 TOP5 标签与避让算法)
# =================================================================
original_view_func = py3Dmol.view

def patched_view(*args, **kwargs):
    v = original_view_func(*args, **kwargs)
    original_show = v.show

    def patched_show(*s_args, **s_kwargs):
        try:
            # 读取数据并提取 TOP 5
            prot_df = pd.read_csv(protein_interpret)
            res_col = 'Residue_ID' if 'Residue_ID' in prot_df.columns else 'residue_index'
            score_col = 'PSICHIC_Residue_Percentile' if 'PSICHIC_Residue_Percentile' in prot_df.columns else 'percentile_ranking'

            if res_col in prot_df.columns and score_col in prot_df.columns:
                clean_df = prot_df.dropna(subset=[res_col, score_col]).drop_duplicates(subset=[res_col])
                top5 = clean_df.nlargest(5, score_col)

                offsets = [[-45, -45], [45, -45], [-45, 45], [45, 45], [0, -70]]
                extracted_res = []

                for i, (_, row) in enumerate(top5.iterrows()):
                    resi_num = str(int(row[res_col]))
                    resi_name = row['Residue_Type'] if 'Residue_Type' in prot_df.columns else "Res"
                    score_val = row[score_col]

                    extracted_res.append(f"{resi_name}{resi_num}")
                    label_text = f"{resi_name} {resi_num}\n({score_val:.2f})"

                    v.addLabel(
                        label_text,
                        {
                            'fontColor': 'white',
                            'backgroundColor': '#d32f2f',
                            'showBackground': True,
                            'fontsize': 11,
                            'backgroundOpacity': 0.85,
                            'screenOffset': offsets[i],
                            'inFront': True
                        },
                        {'model': 0, 'resi': resi_num}
                    )
                print(f"✨ 成功提取并标注以下 Top 残基: {', '.join(extracted_res)}")

                v.zoomTo({'model': 0})
                v.zoom(0.85)

        except Exception as e:
            print(f"⚠️ 标签生成失败: {e}")

        return original_show(*s_args, **s_kwargs)

    v.show = patched_show
    return v

py3Dmol.view = patched_view

# =================================================================
# 执行原生绘图调用与注入下载组件
# =================================================================
try:
    protein_dict, ligand_dict = plot_visual(
        protein_interpret, ligand_interpret,
        protein_path=protein_path, ligand_path=ligand_path,
        residue_color_type=residue_color,
        show_protein_surface=show_protein_surface,
        show_protein_mainchain=show_protein_mainchain,
        show_protein_sidechain=show_protein_sidechain
    )

    # 注入一键下载 HTML/JS 按钮组件 (增强版抓取逻辑)
    download_ui = """
    <div style="margin-top: 20px; padding: 10px; background: #f8f9fa; border-radius: 8px; border: 1px solid #ddd; display: inline-block;">
        <span style="font-weight: bold; margin-right: 15px;">💾 快速导出：</span>
        <button onclick="download3D()" style="padding: 8px 15px; background: #4CAF50; color: white; border: none; border-radius: 4px; cursor: pointer; box-shadow: 0 2px 4px rgba(0,0,0,0.2);">⬇️ 下载当前视角 3D 截图</button>
        <button onclick="download2D()" style="padding: 8px 15px; background: #2196F3; color: white; border: none; border-radius: 4px; cursor: pointer; margin-left: 10px; box-shadow: 0 2px 4px rgba(0,0,0,0.2);">⬇️ 下载 2D 权重图</button>
    </div>

    <script>
    function download3D() {
        let canvases = document.querySelectorAll('canvas');
        if(canvases.length > 0) {
            let canvas = canvases[canvases.length - 1];
            let url = canvas.toDataURL('image/png');
            let a = document.createElement('a');
            a.href = url;
            a.download = 'PSICHIC_3D_Binding_Site.png';
            a.click();
        } else {
            alert('未检测到 3D 视图，请确保图像已完全加载。');
        }
    }

    function download2D() {
        // 暴力遍历当前输出区域内的所有图片标签
        let imgs = document.querySelectorAll('img');
        let targetImg = null;

        // 倒序查找，因为 matplotlib 画出的图通常是最后一个 img 标签
        for(let i = imgs.length - 1; i >= 0; i--) {
            // 过滤掉可能存在的微小 icon，找到由 python 渲染出的 base64 结果图片
            if(imgs[i].src.startsWith('data:image/png;base64') || imgs[i].src.includes('blob')) {
                targetImg = imgs[i];
                break;
            }
        }

        if(targetImg) {
            let a = document.createElement('a');
            a.href = targetImg.src;
            a.download = 'PSICHIC_2D_Ligand.png';
            a.click();
        } else {
            alert('未检测到 2D 图片组件。\\n\\n💡 提示：\\n1. 请检查代码开头的 ligand_path 是否为空？如果为空，原生代码不会生成 2D 结构图。\\n2. 请确保 2D 图形已经在上方完全显示出来。');
        }
    }
    </script>
    """
    display(HTML(download_ui))

finally:
    py3Dmol.view = original_view_func

protein sequence length: 115
No PDB detected - generating in silico structure from ESM
length 115


GridspecLayout(children=(Output(layout=Layout(grid_area='widget001')), Output(layout=Layout(grid_area='widget0…

In [20]:
#@title 1. 上传并预处理文件
import os
from google.colab import files
from Bio.PDB import MMCIFParser, PDBIO

pair_id = "LYH000011" #@param {type:"string"}

# --- 文件上传 ---
print("👇 请上传您的蛋白质文件 (.pdb 或 .cif):")
uploaded_protein = files.upload()
raw_protein_path = list(uploaded_protein.keys())[0] if uploaded_protein else ""

print("👇 请上传您的配体文件 (.sdf 或 .mol2):")
uploaded_ligand = files.upload()
ligand_path = list(uploaded_ligand.keys())[0] if uploaded_ligand else ""

# --- 核心：修复 CIF 格式并保存为 PDB ---
protein_path = raw_protein_path
if raw_protein_path.lower().endswith('.cif'):
    print("🔄 检测到 CIF 格式，正在转换索引以适配 plot_visual...")
    try:
        parser = MMCIFParser(QUIET=True)
        structure = parser.get_structure("tmp", raw_protein_path)
        # 强制重置模型 ID 为 0，避免 plot_visual 内部 KeyError: 0
        for i, model in enumerate(structure):
            model.id = i
        io = PDBIO()
        io.set_structure(structure)
        protein_path = "fixed_protein.pdb"
        io.save(protein_path)
        print("✅ 转换完成，已生成适配文件。")
    except Exception as e:
        print(f"⚠️ 转换失败: {e}")

👇 请上传您的蛋白质文件 (.pdb 或 .cif):


Saving LXR YES.pdb to LXR YES.pdb
👇 请上传您的配体文件 (.sdf 或 .mol2):


Saving beta-ionone.sdf to beta-ionone.sdf


In [24]:
#@title 2. 可视化 (增强稳定版：强制提取 5 个独立残基)
import os
import io as sys_io
import pandas as pd
from Bio.PDB.PDBIO import PDBIO
import py3Dmol
from IPython.display import display, HTML

# =================================================================
# 1. 魔法修复: 解决 'PDBIO' object has no attribute 'StringIO'
# =================================================================
if not hasattr(PDBIO, 'StringIO'):
    PDBIO.StringIO = lambda self: sys_io.StringIO()

# =================================================================
# 2. 用户控制面板
# =================================================================
#@markdown **蛋白质基础设置**
residue_color = "Percentile Ranking" #@param ["Original", "Percentile Ranking"]
show_protein_mainchain = True #@param {type:"boolean"}
show_protein_sidechain = False #@param {type:"boolean"}

#@markdown **🌟 表面与渲染设置**
show_protein_surface = True #@param {type:"boolean"}
surface_opacity = 0.8 #@param {type:"slider", min:0.0, max:1.0, step:0.1}
show_top5_labels = True #@param {type:"boolean"}

# =================================================================
# 3. 核心黑科技：底层渲染引擎拦截器
# =================================================================
original_view_func = py3Dmol.view

def patched_view(*args, **kwargs):
    v = original_view_func(*args, **kwargs)
    original_show = v.show

    def patched_show(*s_args, **s_kwargs):
        try:
            # --- 1. 隐藏无效配体坐标 ---
            v.setStyle({'model': 1}, {})

            # --- 2. 强化表面颜色映射与透明度控制 ---
            if show_protein_surface:
                v.removeAllSurfaces()
                v.addSurface(py3Dmol.VDW, {
                    'opacity': surface_opacity,
                    'colorscheme': {'prop': 'b', 'gradient': 'rwb'}
                }, {'model': 0})

            # --- 3. 注入 TOP 5 防重叠标注 (增强数据清洗) ---
            if show_top5_labels:
                try:
                    prot_df = pd.read_csv(protein_interpret)
                    res_col = 'Residue_ID' if 'Residue_ID' in prot_df.columns else 'residue_index'
                    score_col = 'PSICHIC_Residue_Percentile' if 'PSICHIC_Residue_Percentile' in prot_df.columns else 'percentile_ranking'

                    if res_col in prot_df.columns and score_col in prot_df.columns:
                        # 核心修复：去除空值，并根据残基ID去重，确保提取的是 5 个独立的氨基酸
                        clean_df = prot_df.dropna(subset=[res_col, score_col]).drop_duplicates(subset=[res_col])
                        top5 = clean_df.nlargest(5, score_col)

                        offsets = [[-45, -45], [45, -45], [-45, 45], [45, 45], [0, -70]]

                        extracted_res = [] # 记录提取到的残基用于打印

                        for i, (_, row) in enumerate(top5.iterrows()):
                            resi_num = str(int(row[res_col]))
                            resi_name = row['Residue_Type'] if 'Residue_Type' in prot_df.columns else "Res"
                            score_val = row[score_col]

                            extracted_res.append(f"{resi_name}{resi_num}")
                            label_text = f"{resi_name} {resi_num}\n({score_val:.2f})"

                            v.addLabel(
                                label_text,
                                {
                                    'fontColor': 'white',
                                    'backgroundColor': '#d32f2f',
                                    'showBackground': True,
                                    'fontsize': 11,
                                    'backgroundOpacity': 0.9,
                                    'screenOffset': offsets[i],
                                    'inFront': True
                                },
                                {'model': 0, 'resi': resi_num}
                            )
                        print(f"✨ 成功提取并标注以下 Top 残基: {', '.join(extracted_res)}")
                except Exception as e_label:
                    print(f"⚠️ Top5 标注失败: {e_label}")

            # --- 4. 调整适宜的缩放倍数 ---
            v.zoomTo({'model': 0})
            v.zoom(0.85)

        except Exception as e:
            pass

        return original_show(*s_args, **s_kwargs)

    v.show = patched_show
    return v

py3Dmol.view = patched_view

# =================================================================
# 4. 执行渲染与注入下载组件
# =================================================================
try:
    _ = pair_id
    protein_interpret = os.path.join(result_path, 'interpretation_result', pair_id, 'protein.csv')
    ligand_interpret = os.path.join(result_path, 'interpretation_result', pair_id, 'ligand.pkl')

    print("⏳ 正在调用原生函数，绘制高质量 3D 核心口袋大图...")

    plot_visual(
        protein_interpret, ligand_interpret,
        protein_path=protein_path, ligand_path=ligand_path,
        residue_color_type=residue_color,
        show_protein_surface=show_protein_surface,
        show_protein_mainchain=show_protein_mainchain,
        show_protein_sidechain=show_protein_sidechain
    )

    download_ui = """
    <div style="margin-top: 20px; padding: 10px; background: #f8f9fa; border-radius: 8px; border: 1px solid #ddd; display: inline-block;">
        <span style="font-weight: bold; margin-right: 15px;">💾 快速导出：</span>
        <button onclick="download3D()" style="padding: 8px 15px; background: #4CAF50; color: white; border: none; border-radius: 4px; cursor: pointer; box-shadow: 0 2px 4px rgba(0,0,0,0.2);">⬇️ 下载当前视角 3D 截图</button>
        <button onclick="download2D()" style="padding: 8px 15px; background: #2196F3; color: white; border: none; border-radius: 4px; cursor: pointer; margin-left: 10px; box-shadow: 0 2px 4px rgba(0,0,0,0.2);">⬇️ 下载 2D 权重图</button>
    </div>

    <script>
    function download3D() {
        let canvases = document.querySelectorAll('canvas');
        if(canvases.length > 0) {
            let canvas = canvases[canvases.length - 1];
            let url = canvas.toDataURL('image/png');
            let a = document.createElement('a');
            a.href = url;
            a.download = 'PSICHIC_Global_Binding_Pocket.png';
            a.click();
        } else {
            alert('未检测到 3D 视图，请确保图像已完全加载。');
        }
    }

    function download2D() {
        let imgs = document.querySelectorAll('.output_subarea img');
        if(imgs.length > 0) {
            let img = imgs[imgs.length - 1];
            let a = document.createElement('a');
            a.href = img.src;
            a.download = 'PSICHIC_2D_Ligand.png';
            a.click();
        } else {
            alert('未检测到 2D 图片组件。');
        }
    }
    </script>
    """
    display(HTML(download_ui))

except NameError:
    print("❌ 找不到变量，请确保您已经先运行了前面的 Cell！")
except Exception as e:
    print(f"⚠️ 运行出现异常: {e}")
finally:
    py3Dmol.view = original_view_func

⏳ 正在调用原生函数，绘制高质量 3D 核心口袋大图...


GridspecLayout(children=(Output(layout=Layout(grid_area='widget001')), Output(layout=Layout(grid_area='widget0…

In [ ]:
#@title 3. OPTIONAL - Download Interpretation Results for PyMol
#@markdown If you are having issues downloading the result archive, try disabling your adblocker and run this cell again. If that fails click on the little folder icon to the left, navigate to file: `jobname.result.zip`, right-click and select \"Download\" (see [screenshot](https://pbs.twimg.com/media/E6wRW2lWUAEOuoe?format=jpg&name=small)).


def save_annotated_proteinPDB(prot_dic,pair_id):
    if not os.path.exists(f"./PyMol_Interpret/{pair_id}"):
        os.makedirs(f"./PyMol_Interpret/{pair_id}")

    output_pdb_file = f"./PyMol_Interpret/{pair_id}/{pair_id}_annotated_protein.pdb"
    with open(output_pdb_file, 'w') as f:
        f.write(prot_dic['protein_pdb_block'])


def get_best_alignment_transform_with_timeout(lig_a, lig_b):
    return rdMolAlign.GetBestAlignmentTransform(lig_a, lig_b)



def save_annotated_ligandPDB(lig_dic, pair_id):
    import rdkit
    from copy import deepcopy
    ## The function works well with the PDBBind datasets

    contribs = lig_dic['atom_scores']
    lig2d = lig_dic['ligand2d']

    ori_lig3d = read_mol(lig_dic['ligand_path'])


    lig3d = rdkit.Chem.rdmolops.RemoveAllHs(read_mol(lig_dic['ligand_path']))
    ori_lig3d = read_mol(lig_dic['ligand_path'])
    from rdkit.Chem import AllChem

    AllChem.EmbedMolecule(lig2d, randomSeed=0)
    try:
        _, _, atom_map = rdMolAlign.GetBestAlignmentTransform(lig2d, lig3d)
    except timeout_decorator.TimeoutError:
        print('skipping ',pair_id, "Function execution took longer than 10 minutes and was terminated.")

    mapping_dict = dict(atom_map)
    lig3d_reordered = Chem.RenumberAtoms(lig3d, [mapping_dict[i] for i in range(lig3d.GetNumAtoms())])
    lig2d_atom_scores = [(atom.GetSymbol(),atom.GetProp('score'))[1] for atom in lig2d.GetAtoms()]
    lig2d_atom_list = [(atom.GetSymbol(),atom.GetProp('score'))[0] for atom in lig2d.GetAtoms()]

    def atom_coord(mol, idx):
        positions = mol.GetConformer().GetAtomPosition(idx)
        return positions.x, positions.y, positions.z
    lig3d_atom_coord = [atom_coord(lig3d_reordered, idx) for idx, atom in enumerate(lig3d_reordered.GetAtoms())]
    lig3d_atom_list = [atom.GetSymbol() for idx, atom in enumerate(lig3d_reordered.GetAtoms())]

    assert lig3d_atom_list == lig2d_atom_list
    coord2score = dict(zip(lig3d_atom_coord,lig2d_atom_scores))

    lig3d_pdb_block = Chem.MolToPDBBlock(ori_lig3d)
    new_pdb = io.StringIO(lig3d_pdb_block)

    import tempfile
    from biopandas.pdb import PandasPdb
    # Read PDB from string
    with tempfile.NamedTemporaryFile(mode='w', suffix='.pdb', delete=False) as temp_pdb:
        temp_pdb.write(lig3d_pdb_block)
        temp_pdb.flush()
        ppdb = PandasPdb().read_pdb(temp_pdb.name)

    def coor2score_func(row_, score_dict):
        try:
            return score_dict[row_.x_coord,row_.y_coord,row_.z_coord]
        except KeyError:
            assert row_.element_symbol == 'H'
            return '0.0'

    ppdb.df['HETATM']['b_factor'] = ppdb.df['HETATM'].apply(lambda x: float(coor2score_func(x,coord2score)),axis=1)

    if not os.path.exists(f"./PyMol_Interpret/{pair_id}"):
        os.makedirs(f"./PyMol_Interpret/{pair_id}")

    output_pdb_file = f"./PyMol_Interpret/{pair_id}/{pair_id}_annotated_ligand.pdb"
    ppdb.to_pdb(path=output_pdb_file, records=None, gz=False, append_newline=True)
    lig2d_atom_list = [i.upper() for i in lig2d_atom_list]
    d2d = dict(pd.DataFrame(list(zip(lig2d_atom_list,lig2d_atom_scores)),columns=['atom','score']).groupby('atom')['score'].apply(list))
    d3d = dict(pd.DataFrame(list(zip(ppdb.df['HETATM']['element_symbol'].tolist(),ppdb.df['HETATM']['b_factor'].tolist())),columns=['atom','score']).groupby('atom')['score'].apply(list))

    check_df = pd.DataFrame({'atom':[i.upper() for i in d2d.keys()]})
    check_df['2dimensional_scores'] = check_df['atom'].apply(lambda x: [float(i) for i in d2d[x.upper()]] )
    check_df['3dimensional_scores'] = check_df['atom'].apply(lambda x: [float(i) for i in d3d[x.upper()]] )
    for s2, s3 in zip(check_df['2dimensional_scores'].tolist(),check_df['3dimensional_scores']):
        if set(s2) != set(s3):
            print("print there are some inconsistencies betwen mapping 2D to 3D be careful of the 3D annotated ligand file..")

    ##### ligand b factors ##### ligand b factors ##### ligand b factors ##### ligand b factors #####

#@title Do you agree? { run: "auto" }
download_annotated_ProteinPDB = True #@param {type:"boolean"}
download_annotated_LigandPDB = True #@param {type:"boolean"}

if download_annotated_ProteinPDB:
    save_annotated_proteinPDB(protein_dict,pair_id)
    download_protein_file = f"./PyMol_Interpret/{pair_id}/{pair_id}_annotated_protein.pdb"
    files.download(download_protein_file)

if download_annotated_LigandPDB:
    if ligand_dict['ligand_pdb_block'] is None:
        print('No Ligand PDB file to start with..')
    else:
        save_annotated_ligandPDB(ligand_dict,pair_id)
        download_ligand_file = f"./PyMol_Interpret/{pair_id}/{pair_id}_annotated_ligand.pdb"
        files.download(download_ligand_file)